In [1]:
import os
import sys
import urllib.request
import zipfile
import tarfile
import subprocess
import pandas as pd

# Go up one level to the project root
os.chdir('..')

# Ensure folder structure
folders = [
    "data/raw/invoices",
    "data/raw/emails", 
    "data/raw/contracts",
    "data/raw/news",
    "data/processed",
    "data/labeled"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"✅ {folder}")

print(f"🐍 Python executable: {sys.executable}")

✅ data/raw/invoices
✅ data/raw/emails
✅ data/raw/contracts
✅ data/raw/news
✅ data/processed
✅ data/labeled
🐍 Python executable: /usr/local/bin/python3


## 1. BBC News Dataset
Source: University College Dublin


In [2]:
url = "http://mlg.ucd.ie/files/datasets/bbc-fulltext.zip"
zip_path = "data/raw/news/bbc-fulltext.zip"

if not os.path.exists(zip_path):
    print("Downloading BBC News dataset...")
    urllib.request.urlretrieve(url, zip_path)
    print("✅ Download complete")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("data/raw/news/")
    print("✅ Extraction complete")
else:
    print("✅ Dataset already downloaded, skipping...")

for folder in os.listdir("data/raw/news/bbc"):
    folder_path = f"data/raw/news/bbc/{folder}"
    if os.path.isdir(folder_path):
        count = len(os.listdir(folder_path))
        print(f"📁 {folder}: {count} files")

✅ Dataset already downloaded, skipping...
📁 entertainment: 386 files
📁 business: 510 files
📁 sport: 511 files
📁 politics: 417 files
📁 tech: 401 files


## 2. Enron Email Dataset
Source: Kaggle (marcelwiechmann/enron-spam-data)

In [3]:
enron_path = "data/raw/emails/enron_spam_data.csv"

if not os.path.exists(enron_path):
    print("Downloading Enron dataset...")
    subprocess.run([
        sys.executable, "-m", "kaggle.cli", "datasets", "download",
        "-d", "marcelwiechmann/enron-spam-data",
        "-p", "data/raw/emails/",
        "--unzip"
    ], check=True)
    print("✅ Download complete")
else:
    print("✅ Dataset already downloaded, skipping...")

df_emails = pd.read_csv(enron_path)
print(f"📧 Total emails: {len(df_emails)}")
print(df_emails.head())

✅ Dataset already downloaded, skipping...
📧 Total emails: 33716
   Unnamed: 0                       Subject  \
0           0  christmas tree farm pictures   
1           1      vastar resources , inc .   
2           2  calpine daily gas nomination   
3           3                    re : issue   
4           4     meter 7268 nov allocation   

                                             Message Spam/Ham        Date  
0                                                NaN      ham  1999-12-10  
1  gary , production from the high island larger ...      ham  1999-12-13  
2             - calpine daily gas nomination 1 . doc      ham  1999-12-14  
3  fyi - see note below - already done .\nstella\...      ham  1999-12-14  
4  fyi .\n- - - - - - - - - - - - - - - - - - - -...      ham  1999-12-14  


## 3. CUAD Contracts Dataset
Source: Kaggle (konradb/atticus-open-contract-dataset-aok-beta)


In [4]:
cuad_path = "data/raw/contracts"

if not os.path.exists(f"{cuad_path}/master_clauses.csv") and not os.path.exists(f"{cuad_path}/CUAD_v1/master_clauses.csv"):
    print("Downloading CUAD dataset...")
    subprocess.run([
        sys.executable, "-m", "kaggle.cli", "datasets", "download",
        "-d", "konradb/atticus-open-contract-dataset-aok-beta",
        "-p", cuad_path,
        "--unzip"
    ], check=True)
    print("✅ Download complete")
else:
    print("✅ Dataset already downloaded, skipping...")

files = os.listdir(cuad_path)
print(f"📄 Files downloaded: {files}")

contract_file = "data/raw/contracts/CUAD_v1/master_clauses.csv"
if not os.path.exists(contract_file):
    contract_file = "data/raw/contracts/master_clauses.csv"

df_contracts = pd.read_csv(contract_file)
print(f"📄 Total contracts: {len(df_contracts)}")
print(df_contracts.head())

✅ Dataset already downloaded, skipping...
📄 Files downloaded: ['CUAD_v1']
📄 Total contracts: 510
                                            Filename  \
0  CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605...   
1  EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...   
2  FulucaiProductionsLtd_20131223_10-Q_EX-10.9_83...   
3  GopageCorp_20140221_10-K_EX-10.1_8432966_EX-10...   
4  IdeanomicsInc_20160330_10-K_EX-10.26_9512211_E...   

                                    Document Name  \
0               ['MARKETING AFFILIATE AGREEMENT']   
1   ['VIDEO-ON-DEMAND CONTENT LICENSE AGREEMENT']   
2  ['CONTENT DISTRIBUTION AND LICENSE AGREEMENT']   
3           ['WEBSITE CONTENT LICENSE AGREEMENT']   
4                   ['CONTENT LICENSE AGREEMENT']   

                         Document Name-Answer  \
0               MARKETING AFFILIATE AGREEMENT   
1   VIDEO-ON-DEMAND CONTENT LICENSE AGREEMENT   
2  CONTENT DISTRIBUTION AND LICENSE AGREEMENT   
3           WEBSITE CONTENT LICENSE AGREEMENT   
4 

## 4. Invoice NER Dataset
**Source:** Kaggle — [`nikitpatel/invoice-ner-dataset`](https://www.kaggle.com/datasets/nikitpatel/invoice-ner-dataset)  
**Format:** CSV with two columns — `Input` (raw invoice text) and `Final_Output` (structured JSON of extracted fields).  
**Usage:** Only `Input` is used for classification; `Final_Output` is kept for reference.  
**Requirement:** `pip install kagglehub[pandas-datasets]` — and a valid Kaggle account (free).


In [5]:
invoice_csv = "data/raw/invoices/converted_invoice_dataset.csv"
invoice_xlsx = "data/raw/invoices/converted_invoice_dataset.xlsx"

# If a pre-existing .xlsx is present but no .csv, convert it once.
if not os.path.exists(invoice_csv) and os.path.exists(invoice_xlsx):
    print(f"Converting {invoice_xlsx} → {invoice_csv}")
    pd.read_excel(invoice_xlsx).to_csv(invoice_csv, index=False)

if not os.path.exists(invoice_csv):
    print("Downloading Invoice NER dataset via kagglehub...")
    try:
        import kagglehub
        from kagglehub import KaggleDatasetAdapter
        import shutil, glob

        # Option A: load directly into a DataFrame and save locally
        df_invoices = kagglehub.load_dataset(
            KaggleDatasetAdapter.PANDAS,
            "nikitpatel/invoice-ner-dataset",
            "",  # loads the first (and only) CSV in the dataset
        )
        df_invoices.to_csv(invoice_csv, index=False)
        print(f"✅ Downloaded and saved to {invoice_csv}")

    except Exception as e:
        # Option B: fall back to kaggle CLI (requires kaggle API token in ~/.kaggle/kaggle.json)
        print(f"kagglehub failed ({e}), falling back to kaggle CLI...")
        import glob, shutil
        subprocess.run([
            sys.executable, "-m", "kaggle.cli", "datasets", "download",
            "-d", "nikitpatel/invoice-ner-dataset",
            "-p", "data/raw/invoices/",
            "--unzip"
        ], check=True)
        # Rename whatever CSV was downloaded to our standard filename
        downloaded = [f for f in glob.glob("data/raw/invoices/*.csv")
                      if "converted_invoice_dataset" not in f]
        if downloaded:
            shutil.move(downloaded[0], invoice_csv)
            print(f"✅ Renamed {os.path.basename(downloaded[0])} → converted_invoice_dataset.csv")
        else:
            print("✅ Download complete")
else:
    print("✅ Dataset already downloaded, skipping...")

df_invoices = pd.read_csv(invoice_csv)
print(f"🧾 Total invoices : {len(df_invoices):,}")
print(f"   Columns        : {df_invoices.columns.tolist()}")
print()
print("First 5 records:")
df_invoices.head()


✅ Dataset already downloaded, skipping...
🧾 Total invoices : 67
   Columns        : ['Input', 'Final_Output']

First 5 records:


,Input,Final_Output
0,Cream and White Simple Minimalist Catering Ser...,"{""TOTAL_AMOUNT"": ""$1000"", ""DUE_AMOUNT"": ""$550""..."
1,Beige Elegant Professional Business Invoice\n\...,"{""INVOICE_NUMBER"": ""#01234"", ""BILLED_TO"": ""Est..."
2,Black and White Clean Modern Invoice\n\nConsul...,"{""BILL_TO"": ""SALFORD & CO."", ""BANK_NAME"": ""Bor..."
3,Black and White Minimalist Business Invoice\n\...,"{""INVOICE_NUMBER"": ""12345"", ""BILLED_TO"": ""Marc..."
4,White Minimalist Business Invoice\n\nSUBTOTALN...,"{""INVOICE_NUMBER"": ""#123456"", ""DATE_ISSUED"": ""..."


## 5. High-Quality Invoice Images
**Source:** Kaggle — [`osamahosamabdellatif/high-quality-invoice-images-for-ocr`](https://www.kaggle.com/datasets/osamahosamabdellatif/high-quality-invoice-images-for-ocr)  
**Format:** JPG/PNG invoice images across multiple industries, currencies, and layouts.  
**Usage:** Diverse test set for qualitative evaluation of the information-extraction pipeline — *not* used for training. Images are run through the OCR path in `src/pdf_loader.py` when evaluated.

In [7]:
invoice_imgs_dir = "data/raw/invoices/high_quality_images"

def _count_images(path):
    if not os.path.isdir(path):
        return 0
    exts = {".jpg", ".jpeg", ".png"}
    return sum(
        1 for root, _, files in os.walk(path)
        for f in files if os.path.splitext(f.lower())[1] in exts
    )

if _count_images(invoice_imgs_dir) == 0:
    os.makedirs(invoice_imgs_dir, exist_ok=True)
    print("Downloading High-Quality Invoice Images via kagglehub...")
    try:
        import kagglehub, shutil
        cache_path = kagglehub.dataset_download(
            "osamahosamabdellatif/high-quality-invoice-images-for-ocr"
        )
        # Flatten any nested subfolders into invoice_imgs_dir.
        for root, _, files in os.walk(cache_path):
            for f in files:
                if os.path.splitext(f.lower())[1] in {".jpg", ".jpeg", ".png"}:
                    src = os.path.join(root, f)
                    dst = os.path.join(invoice_imgs_dir, f)
                    if not os.path.exists(dst):
                        shutil.copy2(src, dst)
        print(f"✅ Downloaded to {invoice_imgs_dir}")
    except Exception as e:
        print(f"kagglehub failed ({e}), falling back to kaggle CLI...")
        subprocess.run([
            sys.executable, "-m", "kaggle.cli", "datasets", "download",
            "-d", "osamahosamabdellatif/high-quality-invoice-images-for-ocr",
            "-p", invoice_imgs_dir,
            "--unzip"
        ], check=True)
else:
    print("✅ Dataset already downloaded, skipping...")

invoice_img_count = _count_images(invoice_imgs_dir)
print(f"🖼️  Total invoice images : {invoice_img_count:,}")

✅ Dataset already downloaded, skipping...
🖼️  Total invoice images : 4,295


In [8]:
news_count = 0
news_root = "data/raw/news/bbc"
if os.path.exists(news_root):
    for folder in os.listdir(news_root):
        folder_path = os.path.join(news_root, folder)
        if os.path.isdir(folder_path):
            news_count += len(os.listdir(folder_path))

email_count       = len(df_emails)   if 'df_emails'   in globals() else 0
contract_count    = len(df_contracts) if 'df_contracts' in globals() else 0
invoice_count     = len(df_invoices)  if 'df_invoices'  in globals() else 0
invoice_img_count = invoice_img_count if 'invoice_img_count' in globals() else 0

print("=" * 50)
print("📊 DATASET COLLECTION SUMMARY")
print("=" * 50)
print(f"📰 News articles    : {news_count:,}")
print(f"📧 Emails           : {email_count:,}")
print(f"📄 Contracts        : {contract_count:,}")
print(f"🧾 Invoices (text)  : {invoice_count:,} (English Western-style, text-based)")
print(f"🖼️  Invoice images   : {invoice_img_count:,} (diverse eval set, no labels)")
print("=" * 50)
print("✅ All datasets collected successfully!")
print("➡️  Next step: 02_EDA.ipynb")

📊 DATASET COLLECTION SUMMARY
📰 News articles    : 2,225
📧 Emails           : 33,716
📄 Contracts        : 510
🧾 Invoices (text)  : 67 (English Western-style, text-based)
🖼️  Invoice images   : 4,295 (diverse eval set, no labels)
✅ All datasets collected successfully!
➡️  Next step: 02_EDA.ipynb
